# 02 — Computing forest structure products

This notebook covers the full processing pipeline from a TileDB point-cloud array to gridded raster products:

- Canopy Height Model (CHM), DTM, DSM
- Gap fraction and effective LAI
- LiDAR structural metrics
- Aboveground biomass — Næsset model and scikit-learn wrapper
- Reading results as an xarray Dataset
- Multi-temporal change detection

All outputs are written to an `ALSZarrStore` — no GeoTIFF intermediates, no mosaic step.

**Paths** — replace `/path/to/...` with your actual paths.

## Setup

In [ ]:
from alsdb import ALSProvider
from alsdb.storage import ALSZarrStore

reader = ALSProvider(storage_type="local", uri="/path/to/my_array")
store = ALSZarrStore("/path/to/forest.zarr")  # created automatically on first write

BBOX = (308_000, 4_688_000, 310_000, 4_690_000)  # (xmin, ymin, xmax, ymax) in CRS units
YEAR = 2021

## Canopy Height Model / DTM / DSM

`compute_all` computes DTM, DSM, and CHM in a single tiled pass.  
Use `tile_size` and `n_workers` to parallelise over large areas.

A 50 m `tile_buffer` is used for the TileDB query so the Delaunay TIN for HAG is well-constrained at tile edges. Only the non-overlapping `crop_bbox` is written to the Zarr store.

In [ ]:
from alsdb.processing.chm import compute_all

compute_all(
    provider=reader,
    store=store,
    resolution=1.0,
    bbox=BBOX,
    year=YEAR,
    tile_size=500.0,  # 500 m sub-tiles
    tile_buffer=50.0,  # 50 m overlap buffer for accurate HAG at tile edges
    n_workers=4,
)

## Gap fraction and effective LAI

Gap fraction uses the MacArthur–Wilson first-return estimator:

$$P_{gap} = \frac{N_{ground,first}}{N_{ground,first} + N_{veg,first}}$$

Effective LAI is opt-in via `lai=True` using the Beer–Lambert inversion:

$$L_e = -\frac{\ln(P_{gap})}{k}$$

In [ ]:
from alsdb.processing.gap import compute_gap_fraction

compute_gap_fraction(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    lai=True,
    k=0.5,  # extinction coefficient (spherical leaf angle distribution)
    tile_size=500.0,
    n_workers=4,
)

## LiDAR structural metrics

`compute_metrics` writes h50, h75, h95, hmean, canopy cover (`cc`), and point density to the store.
These are used as predictors for the biomass models below.

In [ ]:
from alsdb.processing.biomass import compute_metrics

compute_metrics(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    tile_size=500.0,
    n_workers=4,
)

## Aboveground biomass — Næsset (2002) model

The default allometric model is:

$$\text{AGB} = a \cdot h_{95}^{\,b} \cdot cc^{\,c}$$

Default parameters: `a=0.8, b=1.8, c=0.5`.  
> **Note:** Calibrate against field inventory plots before using the output scientifically.

In [ ]:
from alsdb.processing.biomass import compute_biomass

compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    tile_size=500.0,
    n_workers=4,
)

You can also pass a custom allometric function directly:

In [ ]:
def my_model(metrics):
    return 1.2 * metrics["h95"] ** 2.1 * metrics["cc"] ** 0.6


compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    model_fn=my_model,
)

## Aboveground biomass — scikit-learn model

`wrap_sklearn_model` wraps any trained sklearn-compatible estimator (Random Forest, Gradient Boosting, Ridge, Pipeline, …).  
It handles reshaping the per-cell metric grids into the `(n_samples, n_features)` matrix sklearn expects and masks NaN pixels automatically.

Default feature order: `["h50", "h75", "h95", "hmean", "cc", "density"]` — use the `features` argument to specify a subset.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from alsdb.processing.biomass import wrap_sklearn_model

# X_train columns must match the feature order (h50, h75, h95, hmean, cc, density)
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_agb)

compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    model_fn=wrap_sklearn_model(rf),
)

In [ ]:
# Use only a subset of features (must match the order used during training)
compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    model_fn=wrap_sklearn_model(rf, features=["h95", "cc", "density"]),
)

## Reading results as xarray

`store.to_dataset(resolution)` opens a resolution group as a CRS-aware `xarray.Dataset` (via rioxarray if available).  
The time axis corresponds to survey year.

In [ ]:
ds1m = store.to_dataset(resolution=1.0)
ds10m = store.to_dataset(resolution=10.0)

chm = ds1m["chm"].sel(time=YEAR)  # (ny, nx)
dtm = ds1m["dtm"].sel(time=YEAR)
agb = ds10m["biomass"].sel(time=YEAR)
h95 = ds10m["h95"].sel(time=YEAR)
gap = ds10m["gap"].sel(time=YEAR)

print(ds1m)
print("CRS:", ds1m.rio.crs)

## Multi-temporal change detection

Store all survey years in the same Zarr store — the time axis handles them.

In [ ]:
for year in [2017, 2021, 2023]:
    compute_all(
        provider=reader,
        store=store,
        resolution=1.0,
        bbox=BBOX,
        year=year,
        tile_size=500.0,
        n_workers=4,
    )

ds = store.to_dataset(resolution=1.0)
delta_chm = ds["chm"].sel(time=2021) - ds["chm"].sel(time=2017)
print("Max canopy height gain 2017→2021:", float(delta_chm.max()), "m")